# Streaming Tokens to the User: SSE, WebSockets, and the Wait the Reader Feels

The full code of [Streaming Tokens to the User: SSE, WebSockets, and the Wait the Reader Feels](https://residual-stream.lvtan71.workers.dev/posts/streaming-tokens-sse-websockets/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/streaming-tokens-sse-websockets) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/streaming-tokens-sse-websockets.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 2 min end to end on the RTX 3090 machine that saved the outputs below.
Downloads Qwen2.5-1.5B-Instruct (about 3 GB). Generated text differs by machine, so byte counts move slightly.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-17** (torch 2.14.1+cu130, 2 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "torch==2.14.*" "transformers==5.17.*" huggingface_hub

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "streaming-tokens-sse-websockets"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/streaming-tokens-sse-websockets


## `split_chars.py`

A Vietnamese sentence with two emoji, decoded one token at a time and incrementally: which pieces come out broken.

In [2]:
!{sys.executable} split_chars.py

20 tokens; decoded one by one:
  Con |  v | ẹ | t |  � | � | � |  nói | : |  mô |  hình |  sinh |  văn |  bản |  từng |  token |  một |  � | � | �
incremental:
  Con |  v | ẹ | t |  |  |  🦜 |  nói | : |  mô |  hình |  sinh |  văn |  bản |  từng |  token |  một |  |  |  🫠
broken pieces one by one: 6, incremental: 0; joined text equal to the original: True


The post printed:

```text
20 tokens; decoded one by one:
  Con |  v | ẹ | t |  � | � | � |  nói | : |  mô |  hình |  sinh |  văn |  bản |  từng |  token |  một |  � | � | �
incremental:
  Con |  v | ẹ | t |  |  |  🦜 |  nói | : |  mô |  hình |  sinh |  văn |  bản |  từng |  token |  một |  |  |  🫠
broken pieces one by one: 6, incremental: 0; joined text equal to the original: True
```

## `measure.py`

Starts `server.py` in the background and sends five prompts from Decoding, 100 tokens each, as one JSON response, as server-sent events and over a WebSocket: time to the first text and to the end, and bytes on the wire.

In [3]:
!{sys.executable} measure.py

Unhandled exception in client_connected_cb
transport: <_SelectorSocketTransport fd=37 read=idle write=<idle, bufsize=0>>
Traceback (most recent call last):
  File "residual-stream-code/posts/streaming-tokens-sse-websockets/server.py", line 49, in handle
    head = (await reader.readuntil(b"\r\n\r\n")).decode()
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/asyncio/streams.py", line 657, in readuntil
    raise exceptions.IncompleteReadError(chunk, None)
asyncio.exceptions.IncompleteReadError: 0 bytes read on a total of undefined expected bytes


/complete  first text after  2.66 s, all of it after  2.66 s;    602 bytes on the wire per answer, for  495 bytes of text (1.2x)
/stream    first text after  0.03 s, all of it after  2.41 s;   2632 bytes on the wire per answer, for  495 bytes of text (5.3x)
/ws        first text after  0.03 s, all of it after  2.41 s;   2044 bytes on the wire per answer, for  495 bytes of text (4.1x)
the same text all three ways: True


The post printed:

```text
/complete  first text after  4.68 s, all of it after  4.68 s;    596 bytes on the wire per answer, for  488 bytes of text (1.2x)
/stream    first text after  0.13 s, all of it after  4.75 s;   2626 bytes on the wire per answer, for  488 bytes of text (5.4x)
/ws        first text after  0.13 s, all of it after  5.12 s;   2038 bytes on the wire per answer, for  488 bytes of text (4.2x)
the same text all three ways: True
```

## `event_bytes.py`

What one streamed piece of text costs on the wire as this post's SSE event, Anthropic's and OpenAI's event formats, and a WebSocket frame.

In [4]:
!{sys.executable} event_bytes.py

this post's SSE event             23 bytes for 3 bytes of text
Anthropic content_block_delta    118 bytes for 3 bytes of text
OpenAI chat.completion.chunk     257 bytes for 3 bytes of text
this post's WebSocket frame       17 bytes for 3 bytes of text


The post printed:

```text
this post's SSE event             23 bytes for 3 bytes of text
Anthropic content_block_delta    118 bytes for 3 bytes of text
OpenAI chat.completion.chunk     257 bytes for 3 bytes of text
this post's WebSocket frame       17 bytes for 3 bytes of text
```

## `reading_speed.py`

Tokens per word on the blog's prose (the 20 human continuations in `prompts.json`) against a reader's 238 words a minute, and what a reader waits for with and without streaming.

In [5]:
!{sys.executable} reading_speed.py

1.38 tokens per word; a reader at 238 words a minute takes in 5.5 tokens a second; this laptop generates 21, 3.9 times as fast
a 50-word answer (69 tokens): 3.2 s to generate, 13 s to read; without streaming the reader waits 3.2 s before the first word, with it about 0.13 s
a 300-word answer (415 tokens): 19.5 s to generate, 76 s to read; without streaming the reader waits 19.5 s before the first word, with it about 0.13 s


The post printed:

```text
1.38 tokens per word; a reader at 238 words a minute takes in 5.5 tokens a second; this laptop generates 21, 3.9 times as fast
a 50-word answer (69 tokens): 3.2 s to generate, 13 s to read; without streaming the reader waits 3.2 s before the first word, with it about 0.13 s
a 300-word answer (415 tokens): 19.5 s to generate, 76 s to read; without streaming the reader waits 19.5 s before the first word, with it about 0.13 s
```

## `experiments.py`

Against a server it starts and restarts: an SSE client that leaves after 10 pieces (with the server watching its connection, then with `--naive`), a WebSocket client that says stop, and the same prompt through `proxy.py`, buffering and then with `--honor`.

In [6]:
!{sys.executable} experiments.py

Unhandled exception in client_connected_cb
transport: <_SelectorSocketTransport fd=37 read=idle write=<idle, bufsize=0>>
Traceback (most recent call last):
  File "residual-stream-code/posts/streaming-tokens-sse-websockets/server.py", line 49, in handle
    head = (await reader.readuntil(b"\r\n\r\n")).decode()
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/asyncio/streams.py", line 657, in readuntil
    raise exceptions.IncompleteReadError(chunk, None)
asyncio.exceptions.IncompleteReadError: 0 bytes read on a total of undefined expected bytes


SSE client left after 10 pieces: the server generated 10 tokens in all


Unhandled exception in client_connected_cb
transport: <_SelectorSocketTransport fd=37 read=idle write=<idle, bufsize=0>>
Traceback (most recent call last):
  File "residual-stream-code/posts/streaming-tokens-sse-websockets/server.py", line 49, in handle
    head = (await reader.readuntil(b"\r\n\r\n")).decode()
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/asyncio/streams.py", line 657, in readuntil
    raise exceptions.IncompleteReadError(chunk, None)
asyncio.exceptions.IncompleteReadError: 0 bytes read on a total of undefined expected bytes


socket.send() raised exception.


socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.


...

SSE client left after 10 pieces: the server generated 100 tokens in all


Unhandled exception in client_connected_cb
transport: <_SelectorSocketTransport fd=37 read=idle write=<idle, bufsize=0>>
Traceback (most recent call last):
  File "residual-stream-code/posts/streaming-tokens-sse-websockets/server.py", line 49, in handle
    head = (await reader.readuntil(b"\r\n\r\n")).decode()
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/asyncio/streams.py", line 657, in readuntil
    raise exceptions.IncompleteReadError(chunk, None)
asyncio.exceptions.IncompleteReadError: 0 bytes read on a total of undefined expected bytes


WebSocket client said stop after 10 pieces: the server generated 10 tokens in all


through the proxy: first text after 2.44 s, the end after 2.44 s, 100 pieces, 2,679 bytes
Unhandled exception in client_connected_cb
transport: <_SelectorSocketTransport fd=37 read=idle write=<idle, bufsize=0>>
Traceback (most recent call last):
  File "residual-stream-code/posts/streaming-tokens-sse-websockets/server.py", line 49, in handle
    head = (await reader.readuntil(b"\r\n\r\n")).decode()
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/asyncio/streams.py", line 657, in readuntil
    raise exceptions.IncompleteReadError(chunk, None)
asyncio.exceptions.IncompleteReadError: 0 bytes read on a total of undefined expected bytes


through the proxy: first text after 0.03 s, the end after 2.54 s, 100 pieces, 2,679 bytes


The post printed:

```text
SSE client left after 10 pieces: the server generated 10 tokens in all
SSE client left after 10 pieces: the server generated 100 tokens in all
WebSocket client said stop after 10 pieces: the server generated 10 tokens in all
through the proxy: first text after 4.91 s, the end after 4.91 s, 100 pieces, 2,583 bytes
through the proxy: first text after 0.20 s, the end after 5.03 s, 100 pieces, 2,583 bytes
```